In [ ]:
import json
import sys
import os
import numpy as np
import pandas as pd
import random
import torch
from torch.utils.data import DataLoader
from collections import defaultdict

models_path = os.path.abspath(os.path.join('..', 'models'))
sys.path.append(models_path)

models_path = os.path.abspath(os.path.join('..', 'src'))
sys.path.append(models_path)

models_path = os.path.abspath(os.path.join('..', 'data'))
sys.path.append(models_path)

from answer_set import AnswerSet
from collate_batch import collate_batch
from DKT.dkt_k_fold import k_fold_cv_dkt
from DKT.dkt_train import train_dkt


In [ ]:
df_answers = pd.read_csv('../data/preprocessed/answers_df.csv')

num_problems = df_answers['problem_id'].nunique()
df_answers['problem_id_reindexed'] = df_answers['problem_id'].rank(method='dense').astype(int)
print(f"Number of unique problems: {num_problems}")

dict_answers = defaultdict(list)

# Group by 'user_id' and iterate through each group
for user_id, user_group in df_answers.groupby('user_id'):
    dict_answers[user_id] = list(zip(user_group['problem_id_reindexed'], user_group['correct']))



Number of unique problems: 8209


The number of unique exercises is too high, we need random vector representations.
Based on the paper we should transform to the following dimension:

In [130]:
embed_dim = round(np.log(num_problems))
print(embed_dim)

9


# Set constants

In [131]:
NUM_EPOCHS = 15
BATCH_SIZE = 100
NUM_FOLDS = 5  # Number of folds for cross-validation

train_ratio = 0.8  # 80% for training, 20% for testing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Set configs

In [132]:
configs = [
    {
        "learning_rate": lr,
        "models_params": {
            "num_items": num_problems,
            "embed_dim": embed_dim,
            "hid_size": 200,
            "num_hid_layers": num_hid_layers,
            "drop_prob": drop_prob,
        },
    }
    for lr in [1e-3, 1e-4, 1e-5]  # 3 reasonable options for learning rate
    for num_hid_layers in [1, 2]  # Hidden layers 1 or 2
    for drop_prob in [0.3, 0.4, 0.5]  # Dropout rate 0.3, 0.4, 0.5
]

# Example: Printing configurations
for idx, config in enumerate(configs, 1):
    print(f"Config {idx}:\n{config}\n")



Config 1:
{'learning_rate': 0.001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.3}}

Config 2:
{'learning_rate': 0.001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.4}}

Config 3:
{'learning_rate': 0.001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.5}}

Config 4:
{'learning_rate': 0.001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.3}}

Config 5:
{'learning_rate': 0.001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.4}}

Config 6:
{'learning_rate': 0.001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.5}}

Config 7:
{'learning_rate': 0.0001, 'models_params': {'num_items': 8209, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 

# Split data into train and test sets

In [133]:

# Get all keys and shuffle them
keys = list(dict_answers.keys())
random.shuffle(keys)

# Split keys into train and test
split_index = int(len(keys) * train_ratio)
train_keys = keys[:split_index]
test_keys = keys[split_index:]

# Create train and test dictionaries
train_dict = {key: dict_answers[key] for key in train_keys}
test_dict = {key: dict_answers[key] for key in test_keys}


# Hyperparameter-tuning

In [136]:
best_val_auc_avg = 0  # Track the best validation AUC
best_config = None  # Track the best configuration

for idx, config in enumerate(configs, 1):
    print(f"\nEvaluating Config {idx}/{len(configs)}")

    lr = config['learning_rate']
    model_params = config['models_params']

    val_auc_avg = k_fold_cv_dkt(
        num_folds=NUM_FOLDS,
        model_params=model_params,
        lr=lr,
        num_epochs=NUM_EPOCHS,
        device=device,
        train_dict=train_dict,
        batch_size=BATCH_SIZE,
        collate_batch=collate_batch,
        )

    # Update the global best if needed
    if val_auc_avg > best_val_auc_avg:
        best_val_auc_avg = val_auc_avg
        best_config = config  # Save the best configuration
        print(f"\nNew best model found: Config {idx}. Validation AUC: {best_val_auc_avg:.4f}")

print(f"\nBest test AUC: {best_val_auc_avg:.4f}")
print(f"\nBest Configuration: {best_config}")


A streamkimeneten csak az utolsó 5000 sor látható.

Epoch 8

Training: 23 batches [00:03,  7.17 batches/s]

Epoch 9

Training: 23 batches [00:02,  8.88 batches/s]

Epoch 10

Training: 23 batches [00:02,  8.78 batches/s]

Epoch 11

Training: 23 batches [00:02,  9.03 batches/s]

Epoch 12

Training: 23 batches [00:02,  8.77 batches/s]

Epoch 13

Training: 23 batches [00:03,  7.05 batches/s]

Epoch 14

Training: 23 batches [00:02,  9.05 batches/s]

Epoch 15

Training: 23 batches [00:02,  8.95 batches/s]
Evaluation: 6 batches [00:00,  9.64 batches/s]
For the 3. fold AUC is 0.842556052866619.

Fold 4/5

Epoch 1

Training: 23 batches [00:02,  8.59 batches/s]

Epoch 2

Training: 23 batches [00:02,  8.01 batches/s]

Epoch 3

Training: 23 batches [00:03,  7.38 batches/s]

Epoch 4

Training: 23 batches [00:02,  8.84 batches/s]

Epoch 5

Training: 23 batches [00:02,  8.87 batches/s]

Epoch 6

Training: 23 batches [00:02,  8.78 batches/s]

Epoch 7

Training: 23 batches [00:03,  7.50 batches/s]

Epo

In [146]:
# training on the whole train set
train_dict = dict(sorted(train_dict.items(), key=lambda item: len(item[1])))
train_dataset = AnswerSet(train_dict)
train_loader = DataLoader(train_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=True, num_workers=2)

test_dict = dict(sorted(test_dict.items(), key=lambda item: len(item[1])))
test_dataset = AnswerSet(test_dict)
test_loader = DataLoader(test_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=True, num_workers=2)


In [148]:
best_model, test_auc = train_dkt(
    model_params=best_config['models_params'],
    lr=best_config['learning_rate'],
    num_epochs=NUM_EPOCHS,
    device=device,
    train_loader=train_loader,
    val_loader=test_loader
)


Epoch 1

Training: 28 batches [00:03,  8.44 batches/s]

Epoch 2

Training: 28 batches [00:03,  7.16 batches/s]

Epoch 3

Training: 28 batches [00:03,  8.50 batches/s]

Epoch 4

Training: 28 batches [00:03,  8.69 batches/s]

Epoch 5

Training: 28 batches [00:03,  8.46 batches/s]

Epoch 6

Training: 28 batches [00:03,  7.02 batches/s]

Epoch 7

Training: 28 batches [00:03,  8.58 batches/s]

Epoch 8

Training: 28 batches [00:03,  8.69 batches/s]

Epoch 9

Training: 28 batches [00:03,  8.46 batches/s]

Epoch 10

Training: 28 batches [00:04,  6.10 batches/s]

Epoch 11

Training: 28 batches [00:03,  8.69 batches/s]

Epoch 12

Training: 28 batches [00:03,  8.67 batches/s]

Epoch 13

Training: 28 batches [00:03,  8.60 batches/s]

Epoch 14

Training: 28 batches [00:04,  6.92 batches/s]

Epoch 15

Training: 28 batches [00:03,  8.72 batches/s]
Evaluation: 7 batches [00:00,  9.14 batches/s]


In [ ]:
print(f"The achieved test AUC: {test_auc}")

model_save_path = "../data/models/best_dkt_model.pth"
torch.save(best_model.state_dict(), model_save_path)
print(f"Best model saved to {model_save_path}")

# Save the best configuration
config_save_path = "../data/models/best_dkt_config.json"
with open(config_save_path, "w") as f:
    json.dump(best_config, f, indent=4)
print(f"Best configuration saved to {config_save_path}")


Best model saved to best_dkt_model.pth
Best configuration saved to best_dkt_config.json
